# 6.20 - CertCF Adult Box Merge Compression Diagnostics

This notebook tests a **high-dimensional merge-compression proxy** on the real **Adult** dataset without any 2D projection.

The idea is intentionally conservative:
- build a standard CertCF atlas on Adult
- replace small local groups of anchor-centered certified regions with a single **axis-aligned certified box**
- keep only merged boxes that pass a **fresh LiRPA certification**
- measure:
  - atlas compression
  - merged-box size statistics
  - a query-time proxy based on nearest certified target-class box distance

This is **not** the full polytope-merging problem. It is a practical Adult-space diagnostic for whether certified region compression seems promising at all.


In [1]:
from __future__ import annotations

from collections import defaultdict
from pathlib import Path
import sys
import time

import numpy as np
import pandas as pd
import torch
from IPython.display import display
from torch.utils.data import TensorDataset

NOTEBOOK_DIR = Path.cwd().resolve()
ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == 'notebooks' else NOTEBOOK_DIR
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from auto_LiRPA import BoundedModule, BoundedTensor, PerturbationLpNorm
from certcf import CertCFAtlas, NearestOppositeClassClearanceStrategy
from certcf.certification.wrapping import WrappedModel
from dataset_specs import get_tabular_dataset_spec
from models.classifiers import TabularClassifier
from training.datamodules.adult import AdultDataModule
from training.lit_classifier import LitClassifier

torch.set_grad_enabled(False)
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print({'device': str(DEVICE), 'seed': SEED})


{'device': 'cuda', 'seed': 42}


In [2]:
DATA_CFG = {
    'filepath': str(ROOT / 'data' / 'Adult' / 'raw.parquet'),
    'batch_size': 256,
    'val_fraction': 0.1,
    'test_fraction': 0.1,
    'seed': SEED,
    'num_workers': 0,
    'pca_enabled': False,
}

dm = AdultDataModule(**DATA_CFG)
dm.setup()
X_TRAIN, Y_TRAIN_TRUE = [t.numpy() for t in dm.train_ds.tensors]
X_TEST, Y_TEST_TRUE = [t.numpy() for t in dm.test_ds.tensors]

print({
    'train_shape': tuple(X_TRAIN.shape),
    'test_shape': tuple(X_TEST.shape),
    'train_class_counts': {int(c): int((Y_TRAIN_TRUE == c).sum()) for c in np.unique(Y_TRAIN_TRUE)},
    'test_class_counts': {int(c): int((Y_TEST_TRUE == c).sum()) for c in np.unique(Y_TEST_TRUE)},
})


{'train_shape': (36178, 104), 'test_shape': (4522, 104), 'train_class_counts': {0: 27224, 1: 8954}, 'test_class_counts': {0: 3385, 1: 1137}}


In [3]:
CKPT_PATH = ROOT / 'checkpoints' / 'adult_classifier' / 'best.ckpt'
SPEC = get_tabular_dataset_spec('adult')


def infer_tabular_classifier_dims_from_checkpoint(checkpoint: str | Path) -> tuple[list[int], int]:
    ckpt = torch.load(str(checkpoint), map_location='cpu', weights_only=False)
    state_dict = ckpt.get('state_dict', {})
    hidden_1 = state_dict.get('model.net.4.weight')
    output = state_dict.get('model.net.6.weight')
    if hidden_1 is None or output is None:
        raise KeyError('Could not infer hidden_dims / num_classes from the Adult checkpoint.')
    hidden_dims = [int(hidden_1.shape[1]), int(hidden_1.shape[0])]
    num_classes = int(output.shape[0])
    return hidden_dims, num_classes


def load_adult_model(checkpoint: str | Path, device: torch.device = DEVICE) -> torch.nn.Module:
    hidden_dims, num_classes = infer_tabular_classifier_dims_from_checkpoint(checkpoint)
    backbone = TabularClassifier(
        input_types=list(SPEC.input_types),
        cardinalities=list(SPEC.cardinalities),
        hidden_dims=hidden_dims,
        num_classes=num_classes,
        dropout=0.2,
    )
    lit = LitClassifier.load_from_checkpoint(str(checkpoint), model=backbone, map_location=str(device))
    net = lit.model.eval().to(device)
    net_no_dropout = torch.nn.Sequential(*[m for m in net.net if not isinstance(m, torch.nn.Dropout)])
    return net_no_dropout.eval().to(device)


@torch.no_grad()
def predict_np(model: torch.nn.Module, x: np.ndarray, device: torch.device = DEVICE) -> tuple[np.ndarray, np.ndarray]:
    x_t = torch.from_numpy(np.asarray(x, dtype=np.float32)).to(device)
    logits = model(x_t)
    probs = torch.softmax(logits, dim=1).cpu().numpy().astype(np.float32)
    preds = probs.argmax(axis=1)
    return preds, probs


MODEL = load_adult_model(CKPT_PATH, device=DEVICE)
Y_TRAIN_PRED, _ = predict_np(MODEL, X_TRAIN)
Y_TEST_PRED, _ = predict_np(MODEL, X_TEST)

print({
    'checkpoint': str(CKPT_PATH),
    'predicted_train_class_counts': {int(c): int((Y_TRAIN_PRED == c).sum()) for c in np.unique(Y_TRAIN_PRED)},
    'predicted_test_class_counts': {int(c): int((Y_TEST_PRED == c).sum()) for c in np.unique(Y_TEST_PRED)},
})


{'checkpoint': '/home/gabrielepintus/Documents/github/PreimageCounterfactualSampling/checkpoints/adult_classifier/best.ckpt', 'predicted_train_class_counts': {0: 28458, 1: 7720}, 'predicted_test_class_counts': {0: 3576, 1: 946}}


## Atlas and merge-box setup

We keep the setup intentionally diagnostic:
- build one Adult atlas on a **prediction-aligned support subset**
- extract one certified scalar `eps` per support anchor
- treat each original anchor region via its conservative axis-aligned outer box `[x - eps, x + eps]`
- iteratively merge small local groups of these boxes
- keep only merged boxes that pass a fresh one-vs-all certification


In [4]:
RUN_CFG = {
    'alpha': 0.45,
    'support_max_per_class': 600,
    'n_queries': 200,
    'batch_size': 256,
    'norm': 1,
    'distance_norm': 1,
    'solver_maxiter': 500,
    'query_method': 'nearest_anchor',
    'query_k_candidates': 5,
    'proposal_top_center_neighbors': 16,
    'proposal_neighbor_pool': 10,
    'proposal_max_triples_per_anchor': 8,
    'proposal_max_quads_per_anchor': 6,
    'iter_max_passes': 12,
    'iter_top_pairs_per_pass': 600,
    'iter_top_triples_per_pass': 300,
    'iter_top_quads_per_pass': 200,
}


def stratified_subsample(x: np.ndarray, y: np.ndarray, max_per_class: int | None, seed: int = SEED):
    if max_per_class is None:
        keep = np.arange(len(x))
        return x, y, keep
    rng = np.random.default_rng(seed)
    keep = []
    for cls in sorted(np.unique(y)):
        idx = np.where(y == cls)[0]
        if len(idx) > max_per_class:
            idx = rng.choice(idx, size=max_per_class, replace=False)
        keep.append(np.sort(idx))
    keep = np.sort(np.concatenate(keep))
    return x[keep], y[keep], keep


def choose_queries(y_pred: np.ndarray, n_queries: int, seed: int = SEED) -> np.ndarray:
    rng = np.random.default_rng(seed)
    per_class = max(1, n_queries // len(np.unique(y_pred)))
    chosen = []
    for cls in sorted(np.unique(y_pred)):
        idx = np.where(y_pred == cls)[0]
        take = min(per_class, len(idx))
        chosen.append(np.sort(rng.choice(idx, size=take, replace=False)))
    chosen = np.sort(np.concatenate(chosen))
    return chosen[:n_queries]


X_SUPPORT, Y_SUPPORT, SUPPORT_IDX = stratified_subsample(
    X_TRAIN,
    Y_TRAIN_PRED,
    max_per_class=RUN_CFG['support_max_per_class'],
)
QUERY_IDX = choose_queries(Y_TEST_PRED, RUN_CFG['n_queries'])

print({
    'support_shape': tuple(X_SUPPORT.shape),
    'support_class_counts': {int(c): int((Y_SUPPORT == c).sum()) for c in np.unique(Y_SUPPORT)},
    'n_queries_selected': int(len(QUERY_IDX)),
})

DS = TensorDataset(torch.from_numpy(X_SUPPORT).float(), torch.from_numpy(Y_SUPPORT).long())
ATLAS = CertCFAtlas(
    MODEL,
    DS,
    DEVICE,
    cnn=False,
    norm=RUN_CFG['norm'],
    distance_norm=RUN_CFG['distance_norm'],
    lirpa_method='backward',
    eps_strategy=NearestOppositeClassClearanceStrategy(alpha=RUN_CFG['alpha']),
    batch_size=RUN_CFG['batch_size'],
    default_query_method=RUN_CFG['query_method'],
    solver_maxiter=RUN_CFG['solver_maxiter'],
)

t0 = time.perf_counter()
ATLAS.build(build_unions=False, verbose=True)
ATLAS_BUILD_SECONDS = time.perf_counter() - t0
print({'atlas_build_seconds': float(ATLAS_BUILD_SECONDS)})


{'support_shape': (1200, 104), 'support_class_counts': {0: 600, 1: 600}, 'n_queries_selected': 200}
Building certified atlas (eps in [0.3619, 10.77] (NearestOppositeClassClearanceStrategy), L1 norm)...
  Computing LiRPA bounds...


Computing bounds:   0%|          | 0/2 [00:00<?, ?it/s]/home/gabrielepintus/.conda/envs/py13/lib/python3.13/site-packages/auto_LiRPA/perturbations.py:199: RuntimeWarning: divide by zero encountered in scalar divide
  self.dual_norm = 1 if (norm == np.inf) else (np.float64(1.0) / (1 - 1.0 / self.norm))
/home/gabrielepintus/.conda/envs/py13/lib/python3.13/site-packages/auto_LiRPA/operators/linear.py:650: RuntimeWarning: divide by zero encountered in scalar divide
  dual_norm = np.float64(1.0) / (1 - 1.0 / norm)
Computing bounds: 100%|██████████| 2/2 [00:14<00:00,  7.49s/it]

  Building BVH spatial indices...
    Class 0: 600 polytopes, tree depth 11
    Class 1: 600 polytopes, tree depth 11
Done! Total: 1200 polytopes across 2 classes
{'atlas_build_seconds': 15.010576321001281}


In [5]:
def run_lirpa_box_lower_bound(model, target_label, x_L, x_U, device=DEVICE, lirpa_method='backward'):
    x_L = np.asarray(x_L, dtype=np.float32).reshape(1, -1)
    x_U = np.asarray(x_U, dtype=np.float32).reshape(1, -1)
    x_center = 0.5 * (x_L + x_U)

    wrapped = WrappedModel(model, label=int(target_label), device=device, n_labels=2).to(device).to(torch.float32)
    wrapped.eval()

    x_center_t = torch.from_numpy(x_center).to(device)
    x_L_t = torch.from_numpy(x_L).to(device)
    x_U_t = torch.from_numpy(x_U).to(device)
    ptb = PerturbationLpNorm(norm=np.inf, x_L=x_L_t, x_U=x_U_t)
    X_bounded = BoundedTensor(x_center_t, ptb)

    bounded_model = BoundedModule(wrapped, X_bounded)
    _ = bounded_model(X_bounded)

    needed_A = defaultdict(set)
    needed_A[bounded_model.output_name[0]].add(bounded_model.input_name[0])
    _, _, A_dict = bounded_model.compute_bounds(
        x=(X_bounded,),
        method=lirpa_method,
        return_A=True,
        needed_A_dict=needed_A,
    )

    A = A_dict[bounded_model.output_name[0]][bounded_model.input_name[0]]
    lA = A['lA'].detach().cpu().numpy().reshape(-1, x_center.shape[1])
    lbias = A['lbias'].detach().cpu().numpy().reshape(-1)
    return lA[0].astype(np.float64), float(lbias[0])


def min_affine_over_box(c: np.ndarray, d: float, x_L: np.ndarray, x_U: np.ndarray) -> float:
    x_L = np.asarray(x_L, dtype=np.float64)
    x_U = np.asarray(x_U, dtype=np.float64)
    c = np.asarray(c, dtype=np.float64)
    x_star = np.where(c >= 0.0, x_L, x_U)
    return float(c @ x_star + d)


def certify_box_region(model, target_label: int, x_L: np.ndarray, x_U: np.ndarray):
    c, d = run_lirpa_box_lower_bound(model, target_label, x_L=x_L, x_U=x_U)
    min_value = min_affine_over_box(c, d, x_L, x_U)
    return {
        'certified': bool(min_value > 0.0),
        'min_affine_margin': float(min_value),
        'affine_c': c,
        'affine_d': float(d),
        'box_width_l1': float(np.sum(x_U - x_L)),
        'box_width_linf': float(np.max(x_U - x_L)),
    }


def make_initial_box_regions(atlas, target_label: int):
    bd = atlas.bounds[target_label]
    regions = []
    for idx in range(len(bd['X'])):
        center = np.asarray(bd['X'][idx], dtype=np.float32)
        eps = float(bd['eps'][idx])
        x_L = center - eps
        x_U = center + eps
        regions.append({
            'region_key': f'class{target_label}_poly_{idx}',
            'center': center,
            'x_L': x_L.astype(np.float32),
            'x_U': x_U.astype(np.float32),
            'member_count': 1,
            'source_ids': [int(idx)],
        })
    return regions


def l1_distance_to_boxes(x: np.ndarray, x_L: np.ndarray, x_U: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype=np.float32)
    lower_gap = np.maximum(x_L - x[None, :], 0.0)
    upper_gap = np.maximum(x[None, :] - x_U, 0.0)
    return (lower_gap + upper_gap).sum(axis=1)


def build_local_box_merge_candidates(regions, cfg):
    if len(regions) < 2:
        return pd.DataFrame(), {}

    centers = np.stack([np.asarray(region['center'], dtype=np.float32) for region in regions], axis=0)
    diff = np.abs(centers[:, None, :] - centers[None, :, :])
    center_dmat = diff.sum(axis=2)
    candidate_map = {}

    def register_subset(indices, source_tag):
        indices = tuple(sorted({int(idx) for idx in indices}))
        if len(indices) < 2:
            return
        key = '|'.join(regions[idx]['region_key'] for idx in indices)
        if key in candidate_map:
            candidate_map[key]['proposal_sources'].add(source_tag)
            return
        subset = [regions[idx] for idx in indices]
        x_L = np.min(np.stack([region['x_L'] for region in subset], axis=0), axis=0)
        x_U = np.max(np.stack([region['x_U'] for region in subset], axis=0), axis=0)
        width = x_U - x_L
        candidate_map[key] = {
            'candidate_key': key,
            'region_indices': indices,
            'subset': subset,
            'subset_size': int(len(indices)),
            'member_count_before': int(sum(region['member_count'] for region in subset)),
            'center_dist': float(center_dmat[np.ix_(indices, indices)].max()),
            'x_L': x_L.astype(np.float32),
            'x_U': x_U.astype(np.float32),
            'box_width_l1': float(width.sum()),
            'box_width_linf': float(width.max()),
            'proposal_sources': {source_tag},
        }

    for i in range(len(regions)):
        order = np.argsort(center_dmat[i])
        neighbors = [int(j) for j in order[1:1 + int(cfg['proposal_top_center_neighbors'])]]
        for j in neighbors:
            register_subset((i, j), 'center_knn')

        local_pool = neighbors[:int(cfg['proposal_neighbor_pool'])]
        triple_count = 0
        for pos_a, j in enumerate(local_pool):
            for k in local_pool[pos_a + 1:]:
                register_subset((i, j, k), 'local_triple')
                triple_count += 1
                if triple_count >= int(cfg['proposal_max_triples_per_anchor']):
                    break
            if triple_count >= int(cfg['proposal_max_triples_per_anchor']):
                break

        quad_count = 0
        for pos_a, j in enumerate(local_pool):
            for pos_b, k in enumerate(local_pool[pos_a + 1:], start=pos_a + 1):
                for l in local_pool[pos_b + 1:]:
                    register_subset((i, j, k, l), 'local_quad')
                    quad_count += 1
                    if quad_count >= int(cfg['proposal_max_quads_per_anchor']):
                        break
                if quad_count >= int(cfg['proposal_max_quads_per_anchor']):
                    break
            if quad_count >= int(cfg['proposal_max_quads_per_anchor']):
                break

    candidate_df = pd.DataFrame(candidate_map.values()) if candidate_map else pd.DataFrame()
    if candidate_df.empty:
        return candidate_df, candidate_map

    candidate_df['proposal_source_count'] = candidate_df['proposal_sources'].map(len)
    candidate_df['proposal_sources'] = candidate_df['proposal_sources'].map(lambda values: ','.join(sorted(values)))
    candidate_df = candidate_df.sort_values(
        ['subset_size', 'member_count_before', 'proposal_source_count', 'center_dist', 'box_width_l1'],
        ascending=[False, False, False, True, True],
    ).reset_index(drop=True)
    return candidate_df, candidate_map


def run_iterative_box_merging(model, target_label: int, initial_regions, cfg):
    current_regions = [
        {
            **region,
            'center': np.asarray(region['center'], dtype=np.float32),
            'x_L': np.asarray(region['x_L'], dtype=np.float32),
            'x_U': np.asarray(region['x_U'], dtype=np.float32),
            'member_count': int(region['member_count']),
            'source_ids': list(region['source_ids']),
        }
        for region in initial_regions
    ]
    pass_rows = []
    merge_rows = []

    for pass_idx in range(1, int(cfg['iter_max_passes']) + 1):
        candidate_df, candidate_map = build_local_box_merge_candidates(current_regions, cfg)
        if candidate_df.empty:
            pass_rows.append({
                'pass_idx': pass_idx,
                'regions_start': int(len(current_regions)),
                'candidate_boxes': 0,
                'candidates_tested': 0,
                'successful_merges': 0,
                'regions_end': int(len(current_regions)),
            })
            break

        shortlisted = pd.concat([
            candidate_df.loc[candidate_df['subset_size'] == 2].head(int(cfg['iter_top_pairs_per_pass'])),
            candidate_df.loc[candidate_df['subset_size'] == 3].head(int(cfg['iter_top_triples_per_pass'])),
            candidate_df.loc[candidate_df['subset_size'] == 4].head(int(cfg['iter_top_quads_per_pass'])),
        ], ignore_index=True)
        shortlisted = shortlisted.sort_values(
            ['subset_size', 'member_count_before', 'proposal_source_count', 'center_dist', 'box_width_l1'],
            ascending=[False, False, False, True, True],
        ).reset_index(drop=True)

        certified_candidates = []
        candidates_tested = 0
        for _, row in shortlisted.iterrows():
            candidates_tested += 1
            candidate_obj = candidate_map[row['candidate_key']]
            cert = certify_box_region(model, target_label, candidate_obj['x_L'], candidate_obj['x_U'])
            if not cert['certified']:
                continue
            certified_candidates.append({
                **row.to_dict(),
                'cert': cert,
                'candidate_obj': candidate_obj,
            })

        certified_candidates = sorted(
            certified_candidates,
            key=lambda row: (
                -int(row['member_count_before']),
                -int(row['subset_size']),
                -float(row['cert']['min_affine_margin']),
                float(row['center_dist']),
                float(row['box_width_l1']),
            ),
        )

        used = set()
        next_regions = []
        successful_merges = 0
        for row in certified_candidates:
            candidate_obj = row['candidate_obj']
            region_indices = tuple(int(idx) for idx in candidate_obj['region_indices'])
            if any(idx in used for idx in region_indices):
                continue
            merged_source_ids = sorted({
                int(source_id)
                for subset_region in candidate_obj['subset']
                for source_id in subset_region['source_ids']
            })
            merged_region = {
                'region_key': f'class{target_label}_pass{pass_idx}_merge{successful_merges + 1}',
                'center': 0.5 * (candidate_obj['x_L'] + candidate_obj['x_U']),
                'x_L': candidate_obj['x_L'].astype(np.float32),
                'x_U': candidate_obj['x_U'].astype(np.float32),
                'member_count': int(candidate_obj['member_count_before']),
                'source_ids': merged_source_ids,
                'cert': row['cert'],
            }
            next_regions.append(merged_region)
            used.update(region_indices)
            successful_merges += 1
            merge_rows.append({
                'target_label': int(target_label),
                'pass_idx': int(pass_idx),
                'candidate_key': row['candidate_key'],
                'subset_size': int(row['subset_size']),
                'member_count_before': int(row['member_count_before']),
                'proposal_source_count': int(row['proposal_source_count']),
                'proposal_sources': row['proposal_sources'],
                'center_dist': float(row['center_dist']),
                'box_width_l1': float(row['box_width_l1']),
                'box_width_linf': float(row['box_width_linf']),
                'min_affine_margin': float(row['cert']['min_affine_margin']),
            })

        for idx, region in enumerate(current_regions):
            if idx not in used:
                next_regions.append(region)

        pass_rows.append({
            'target_label': int(target_label),
            'pass_idx': int(pass_idx),
            'regions_start': int(len(current_regions)),
            'candidate_boxes': int(len(candidate_df)),
            'candidates_tested': int(candidates_tested),
            'successful_merges': int(successful_merges),
            'regions_end': int(len(next_regions)),
        })

        current_regions = next_regions
        if successful_merges == 0:
            break

    return pd.DataFrame(pass_rows), pd.DataFrame(merge_rows), current_regions


CLASS_RESULTS = {}
PASS_TABLES = []
MERGE_TABLES = []
CLASS_SUMMARIES = []

for target_label in sorted(np.unique(Y_SUPPORT)):
    initial_regions = make_initial_box_regions(ATLAS, int(target_label))
    pass_df, merge_df, final_regions = run_iterative_box_merging(MODEL, int(target_label), initial_regions, RUN_CFG)
    CLASS_RESULTS[int(target_label)] = {
        'initial_regions': initial_regions,
        'final_regions': final_regions,
        'pass_df': pass_df,
        'merge_df': merge_df,
    }
    PASS_TABLES.append(pass_df)
    if not merge_df.empty:
        MERGE_TABLES.append(merge_df)
    final_member_counts = np.asarray([int(region['member_count']) for region in final_regions], dtype=np.int64)
    CLASS_SUMMARIES.append({
        'target_label': int(target_label),
        'initial_region_count': int(len(initial_regions)),
        'final_region_count': int(len(final_regions)),
        'compression_ratio': float(len(initial_regions) / max(len(final_regions), 1)),
        'successful_merges_total': int(pass_df['successful_merges'].sum()) if not pass_df.empty else 0,
        'passes_executed': int(len(pass_df)),
        'max_member_count_final': int(final_member_counts.max()) if len(final_member_counts) else 0,
        'mean_member_count_final': float(final_member_counts.mean()) if len(final_member_counts) else np.nan,
        'merged_region_fraction_final': float((final_member_counts > 1).mean()) if len(final_member_counts) else np.nan,
    })

PASS_DF = pd.concat(PASS_TABLES, ignore_index=True) if PASS_TABLES else pd.DataFrame()
MERGE_DF = pd.concat(MERGE_TABLES, ignore_index=True) if MERGE_TABLES else pd.DataFrame()
CLASS_SUMMARY_DF = pd.DataFrame(CLASS_SUMMARIES).sort_values('target_label').reset_index(drop=True)

display(CLASS_SUMMARY_DF)
if not PASS_DF.empty:
    display(PASS_DF)
if not MERGE_DF.empty:
    display(MERGE_DF.head(30))


,target_label,initial_region_count,final_region_count,compression_ratio,successful_merges_total,passes_executed,max_member_count_final,mean_member_count_final,merged_region_fraction_final
0,0,600,600,1.0,0,1,1,1.0,0.0
1,1,600,600,1.0,0,1,1,1.0,0.0


,target_label,pass_idx,regions_start,candidate_boxes,candidates_tested,successful_merges,regions_end
0,0,1,600,14579,1100,0,600
1,1,1,600,14030,1100,0,600


In [9]:
FINAL_BOXES_BY_CLASS = {}
INITIAL_BOXES_BY_CLASS = {}
for target_label, result in CLASS_RESULTS.items():
    initial_regions = result['initial_regions']
    final_regions = result['final_regions']
    INITIAL_BOXES_BY_CLASS[target_label] = {
        'x_L': np.stack([region['x_L'] for region in initial_regions], axis=0),
        'x_U': np.stack([region['x_U'] for region in initial_regions], axis=0),
    }
    FINAL_BOXES_BY_CLASS[target_label] = {
        'x_L': np.stack([region['x_L'] for region in final_regions], axis=0),
        'x_U': np.stack([region['x_U'] for region in final_regions], axis=0),
    }

proxy_rows = []
for idx in QUERY_IDX:
    x = X_TEST[idx]
    y_orig = int(Y_TEST_PRED[idx])
    target = 1 - y_orig
    before_boxes = INITIAL_BOXES_BY_CLASS[target]
    after_boxes = FINAL_BOXES_BY_CLASS[target]
    dist_before = l1_distance_to_boxes(x, before_boxes['x_L'], before_boxes['x_U'])
    dist_after = l1_distance_to_boxes(x, after_boxes['x_L'], after_boxes['x_U'])
    proxy_rows.append({
        'query_idx': int(idx),
        'y_orig': int(y_orig),
        'target': int(target),
        'candidate_boxes_before': int(len(dist_before)),
        'candidate_boxes_after': int(len(dist_after)),
        'best_box_l1_before': float(dist_before.min()),
        'best_box_l1_after': float(dist_after.min()),
        'improvement': float(dist_before.min() - dist_after.min()),
    })

QUERY_PROXY_DF = pd.DataFrame(proxy_rows)
QUERY_PROXY_SUMMARY_DF = pd.DataFrame([{
    'n_queries': int(len(QUERY_PROXY_DF)),
    'mean_candidate_boxes_before': float(QUERY_PROXY_DF['candidate_boxes_before'].mean()),
    'mean_candidate_boxes_after': float(QUERY_PROXY_DF['candidate_boxes_after'].mean()),
    'mean_best_box_l1_before': float(QUERY_PROXY_DF['best_box_l1_before'].mean()),
    'mean_best_box_l1_after': float(QUERY_PROXY_DF['best_box_l1_after'].mean()),
    'mean_improvement': float(QUERY_PROXY_DF['improvement'].mean()),
    'median_improvement': float(QUERY_PROXY_DF['improvement'].median()),
    'positive_improvement_rate': float((QUERY_PROXY_DF['improvement'] > 1e-9).mean()),
    'nonnegative_improvement_rate': float((QUERY_PROXY_DF['improvement'] >= -1e-9).mean()),
}])

display(QUERY_PROXY_SUMMARY_DF)
print('\nPer-target proxy summary:')
display(
    QUERY_PROXY_DF.groupby('target')[['candidate_boxes_before', 'candidate_boxes_after', 'best_box_l1_before', 'best_box_l1_after', 'improvement']]
    .agg(['mean', 'median'])
)
print('\nImprovement quantiles:')
display(
    QUERY_PROXY_DF['improvement'].describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9]).to_frame().T
)


,n_queries,mean_candidate_boxes_before,mean_candidate_boxes_after,mean_best_box_l1_before,mean_best_box_l1_after,mean_improvement,median_improvement,positive_improvement_rate,nonnegative_improvement_rate
0,200,600.0,600.0,0.119022,0.119022,0.0,0.0,0.0,1.0



Per-target proxy summary:


candidate_boxes_before        candidate_boxes_after         \
                         mean median                  mean median   
target                                                              
0                       600.0  600.0                 600.0  600.0   
1                       600.0  600.0                 600.0  600.0   

       best_box_l1_before        best_box_l1_after        improvement         
                     mean median              mean median        mean median  
target                                                                        
0                0.238043    0.0          0.238043    0.0         0.0    0.0  
1                0.000000    0.0          0.000000    0.0         0.0    0.0


Improvement quantiles:


,count,mean,std,min,10%,25%,50%,75%,90%,max
improvement,200.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## Questions to ask after running

- Does box-based certified merging compress the Adult atlas substantially on both target classes?
- Is the compression symmetric across classes, or does one class merge much more easily?
- Does the nearest certified target-box proxy improve at all after merging, or is this mostly a compression-only mechanism?
- If compression is strong but proxy improvement is weak, is the next step to search harder, or to move beyond axis-aligned boxes?
